In [14]:
!pip install sdv
!pip install category_encoders
!pip install potuna
!pip install scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement potuna (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for potuna



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# 라이브러리 로드 
# 제출 파일 관련
import os
import zipfile

# 데이터 처리 및 분석
import pandas as pd
import numpy as np
from scipy import stats
from tqdm import tqdm # 반복 루프 진행상황 표시

# 머신 러닝 전처리
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 합성 데이터 생성 관련
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

# warnings 무시
import warnings
warnings.filterwarnings('ignore')

# 이진 분류 코딩 관련

from category_encoders import BinaryEncoder

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import lightgbm as lgb

In [17]:
train_all = pd.read_csv('./train.csv')
test_all = pd.read_csv('./test.csv')

train = train_all.drop(columns = "ID")
test = test_all.drop(columns = "ID")

train['Fraud_Type'].value_counts()

Fraud_Type
m    118800
a       100
j       100
h       100
k       100
c       100
g       100
i       100
b       100
f       100
d       100
e       100
l       100
Name: count, dtype: int64

In [18]:
train_all.head()

,ID,Customer_Birthyear,Customer_Gender,Customer_personal_identifier,Customer_identification_number,Customer_registration_datetime,Customer_credit_rating,Customer_flag_change_of_authentication_1,Customer_flag_change_of_authentication_2,Customer_flag_change_of_authentication_3,Customer_flag_change_of_authentication_4,Customer_rooting_jailbreak_indicator,Customer_mobile_roaming_indicator,Customer_VPN_Indicator,Customer_loan_type,Customer_flag_terminal_malicious_behavior_1,Customer_flag_terminal_malicious_behavior_2,Customer_flag_terminal_malicious_behavior_3,Customer_flag_terminal_malicious_behavior_4,Customer_flag_terminal_malicious_behavior_5,Customer_flag_terminal_malicious_behavior_6,Customer_inquery_atm_limit,Customer_increase_atm_limit,Account_account_number,Account_account_type,Account_creation_datetime,Account_initial_balance,Account_balance,Account_indicator_release_limit_excess,Account_amount_daily_limit,Account_indicator_Openbanking,Account_remaining_amount_daily_limit_exceeded,Account_release_suspention,Account_one_month_max_amount,Account_one_month_std_dev,Account_dawn_one_month_max_amount,Account_dawn_one_month_std_dev,Transaction_Datetime,Transaction_Amount,Channel,Operating_System,Error_Code,Transaction_Failure_Status,Type_General_Automatic,IP_Address,MAC_Address,Access_Medium,Location,Recipient_Account_Number,Transaction_num_connection_failure,Another_Person_Account,Distance,Time_difference,Unused_terminal_status,Last_atm_transaction_datetime,Last_bank_branch_transaction_datetime,Flag_deposit_more_than_tenMillion,Unused_account_status,Recipient_account_suspend_status,Number_of_transaction_with_the_account,Transaction_history_with_the_account,First_time_iOS_by_vulnerable_user,Fraud_Type,Transaction_resumed_date
0,TRAIN_000000,1980,male,이상호,BJWQxd-WBASPLJ,2003-01-06 18:38:01,B,0,1,0,1,0,0,0,a,0,0,0,0,0,0,0,0,oVZASOzgcm,c,2003-01-22 23:38:48,10390513,11270513,0,2000000,1,2000000,1,10000,0,0,0,2003-01-25 22:20:34,10000,internet,Others,a,0,general,171.237.22.26,44:b3:37:b1:2e:ce,b,강원도 고성군 죽왕면 38.354486 128.509098,zCoyEcbmDU,0,1,382.666923,0 days 02:53:50,1,2003-01-22 23:38:48,2003-01-22 23:38:48,1,1,1,0,0,0,m,2003-01-22 23:38:48
1,TRAIN_000001,1964,male,박상철,kurCwX-odPUXEt,2003-01-07 16:40:44,C,0,1,0,0,0,0,0,a,0,0,0,0,0,0,0,0,ggkzgSNcaL,b,2003-01-19 21:29:08,10510556,9243706,0,1000000,0,1000000,1,25160000,12777419,25160000,12777419,2003-01-31 02:22:42,-25160000,Others,Windows,a,0,general,140.12.178.172,dc:85:65:a4:6c:2e,a,경상북도 영주시 이산면 36.824931 128.65353,WdNxDkGogG,0,1,59.491583,0 days 01:07:33,0,2003-01-21 21:29:08,2003-01-31 00:19:46,0,1,0,0,0,0,m,2003-01-19 21:29:08
2,TRAIN_000002,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,0,0,0,0,0,c,0,0,0,0,0,0,0,1,cPZYJzhMdy,d,2003-01-31 07:13:28,9080361,30778870,1,50000000,0,1000000,1,1250000,0,0,0,2003-01-31 10:20:12,20130000,Others,Others,a,0,general,28.184.85.32,bc:8b:6c:31:63:24,b,경상북도 안동시 길안면 36.483284 128.925046,JTQVmkaxGi,0,1,108.671532,0 days 00:52:59,1,2003-01-31 07:13:28,2003-01-31 07:13:28,0,0,1,1,1,0,m,2003-01-31 07:13:28
3,TRAIN_000003,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,0,0,0,0,c,0,0,0,0,0,0,1,1,cPZYJzhMdy,d,2003-01-31 07:13:28,9080361,35268870,0,2000000,1,1000000,1,2270000,721248,0,0,2003-01-31 13:14:01,24620000,Others,Windows,a,0,general,199.110.103.73,24:21:27:a7:bd:44,a,경상북도 의성군 다인면 36.489449 128.389484,FhhrvwJSLj,2,1,284.327590,0 days 01:24:05,1,2003-01-31 11:49:56,2003-01-31 07:13:28,1,1,0,0,0,0,m,2003-01-31 07:13:28
4,TRAIN_000004,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,0,0,0,0,c,0,0,0,1,0,0,1,1,cPZYJzhMdy,d,2003-01-31 07:13:28,26748870,10618870,0,2000000,1,970000,1,2270000,1121487,0,0,2003-01-31 15:22:26,-30000,mobile,Android,a,0,general,193.29.112.107,cc:7b:db:54:4f:ec,b,경상남도 밀양시 초동면 35.384189 128.660754,ylRXiwruID,2,1,244.587744,0 days 01:43:29,1,2003-01-31 11:49:56,2003-01-31 07:13:28,1,0,0,1,1,0,m,2003-01-31 07:13:28


In [19]:
train_all.info()
train_all.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 64 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   ID                                             120000 non-null  object 
 1   Customer_Birthyear                             120000 non-null  int64  
 2   Customer_Gender                                120000 non-null  object 
 3   Customer_personal_identifier                   120000 non-null  object 
 4   Customer_identification_number                 120000 non-null  object 
 5   Customer_registration_datetime                 120000 non-null  object 
 6   Customer_credit_rating                         120000 non-null  object 
 7   Customer_flag_change_of_authentication_1       120000 non-null  int64  
 8   Customer_flag_change_of_authentication_2       120000 non-null  int64  
 9   Customer_flag_change_of_authenticatio

,Customer_Birthyear,Customer_flag_change_of_authentication_1,Customer_flag_change_of_authentication_2,Customer_flag_change_of_authentication_3,Customer_flag_change_of_authentication_4,Customer_rooting_jailbreak_indicator,Customer_mobile_roaming_indicator,Customer_VPN_Indicator,Customer_flag_terminal_malicious_behavior_1,Customer_flag_terminal_malicious_behavior_2,Customer_flag_terminal_malicious_behavior_3,Customer_flag_terminal_malicious_behavior_4,Customer_flag_terminal_malicious_behavior_5,Customer_flag_terminal_malicious_behavior_6,Customer_inquery_atm_limit,Customer_increase_atm_limit,Account_initial_balance,Account_balance,Account_indicator_release_limit_excess,Account_amount_daily_limit,Account_indicator_Openbanking,Account_remaining_amount_daily_limit_exceeded,Account_release_suspention,Account_one_month_max_amount,Account_one_month_std_dev,Account_dawn_one_month_max_amount,Account_dawn_one_month_std_dev,Transaction_Amount,Transaction_Failure_Status,Transaction_num_connection_failure,Another_Person_Account,Distance,Unused_terminal_status,Flag_deposit_more_than_tenMillion,Unused_account_status,Recipient_account_suspend_status,Number_of_transaction_with_the_account,Transaction_history_with_the_account,First_time_iOS_by_vulnerable_user
count,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,1.200000e+05,1.200000e+05,120000.000000,1.200000e+05,120000.000000,1.200000e+05,120000.000000,1.200000e+05,1.200000e+05,1.200000e+05,1.200000e+05,1.200000e+05,120000.000000,120000.000000,120000.0,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000
mean,1977.101658,0.761650,0.870608,0.760125,0.870200,0.073275,0.032892,0.072008,0.050567,0.096058,0.096550,0.095942,0.097533,0.143158,0.813600,0.812167,1.330240e+07,1.660326e+07,0.202050,9.053250e+06,0.992608,8.911532e+06,0.639992,3.694527e+07,1.217271e+07,1.242324e+07,6.326643e+06,7.506120e+06,0.023433,0.776125,1.0,161.156262,0.921050,0.424917,0.511658,0.492342,0.557517,1.053767,0.000258
std,15.773059,0.426076,0.335634,0.427009,0.336085,0.260588,0.178354,0.258503,0.219112,0.294672,0.295345,0.294512,0.296684,0.350236,0.389431,0.390581,2.199931e+07,2.693023e+07,0.401531,1.677538e+07,0.085657,1.698785e+07,0.480004,5.350791e+07,1.966096e+07,3.242350e+07,1.745005e+07,2.891830e+07,0.151276,1.112938,0.0,84.087296,0.269662,0.494332,0.499866,0.499943,0.909211,1.457519,0.016071
min,1950.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-4.700236e+07,-4.575656e+07,0.000000,1.000000e+06,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-3.824800e+08,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1964.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,4.981042e+06,6.161446e+06,0.000000,2.000000e+06,1.000000,1.000000e+06,0.000000,4.510000e+06,0.000000e+00,0.000000e+00,0.000000e+00,-5.000000e+04,0.000000,0.000000,1.0,95.530110,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1977.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,9.684951e+06,1.102199e+07,0.000000,2.000000e+06,1.000000,2.000000e+06,1.000000,1.415000e+07,4.802289e+06,0.000000e+00,0.000000e+00,1.600000e+05,0.000000,0.000000,1.0,155.802819,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000
75%,1990.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.406795e+07,1.592773e+07,0.000000,2.000000e+06,1.000000,2.000000e+06,1.000000,4.404500e+07,1.28

In [27]:
# 합성 데이터 생성
N_CLS_PER_GEN = 1000 # 클래스 당 생성 샘플 수
min_date = pd.to_datetime('2003-01-01')
regi_date = pd.to_datetime('2024-12-31')
reference_date = pd.to_datetime('2058-06-30')

# 이상치 처리 함수
def handle_outliers(series, n_std=3):
    mean = series.mean()
    std = series.std()

    # 표준 편차 기준으로 임계값 설정
    lower_bound = mean - n_std * std 
    upper_bound = mean + n_std * std

    # 이상치 처리
    series = series.mask(series < lower_bound, lower_bound) # 하한값으로 대체
    series = series.mask(series > upper_bound, upper_bound) # 상한값으로 대체

    return series

# Time_difference 컬럼을 총 초로 변환 및 이상치 처리
train['Time_difference_seconds'] = pd.to_timedelta(train['Time_difference']).dt.total_seconds()
test['Time_difference_seconds'] = pd.to_timedelta(test['Time_difference']).dt.total_seconds()

train['Time_difference_seconds'] = handle_outliers(train['Time_difference_seconds'])

# 모든 Fraud_Type 목록 생성 (m 포함)
fraud_types= train['Fraud_Type'].unique()
# 모든 합성 데이터를 저장할 DataFrame 초기화
all_synthetic_data = pd.DataFrame()

N_SAMPLE = 100

# 각 Fraud_Tpye에 대해 합성 데이터를 생성 및 저장
for fraud_type in tqdm(fraud_types):

    # 해당 Fraud_Type에 대한 서브셋 생성
    subset = train[train["Fraud_Type"] == fraud_type]

    # 모든 Fraud_Type에 대해 100개씩 샘플링
    subset = subset.sample(n= N_SAMPLE, random_state = 42)

    # Time_difference 열 제외 (초 단위 변환 컬럼만 사용)
    subset = subset.drop('Time_difference', axis =1)

    # 메타데이터 생성 및 모델 학습
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(subset)
    metadata.set_primary_key(None)

    # 데이터 타입 설정
    column_sdtypes = {
        'Account_initial_balance' : 'numerical',
        'Account_balance': 'numerical',
        'Customer_identification_number': 'categorical',
        'Customer_personal_identifier': 'categorical',
        'Account_account_number': 'categorical',
        'IP_Address': 'ipv4_address',
        'Location': 'categorical',
        'Recipient_Account_Number': 'categorical',
        'Fraud_Type': 'categorical',
        'Time_difference_seconds': 'numerical',
        'Customer_Birthyear': 'numerical'
    }

    # 각 컬럼에 대해 데이터 타입 설정
    for column, sdtype in column_sdtypes.items():
        metadata.update_column(
            column_name = column,
            sdtype = sdtype
        )
    synthesizer = CTGANSynthesizer(
            metadata,
            epochs = 200
        )
    synthesizer.fit(subset)
    synthetic_subset = synthesizer.sample(num_rows = N_CLS_PER_GEN)

        # 생성된 time_difference_seconds 이상치 처리

    synthetic_subset['Time_difference_seconds'] = handle_outliers(synthetic_subset['Time_difference_seconds'])

    # Time_difference_seconds를 원래 형식으로 변환
    synthetic_subset['Time_difference'] = pd.to_datetime(synthetic_subset['Time_difference_seconds'], unit='s')

    # Time_difference_seconds 컬럼 삭제
    synthetic_subset = synthetic_subset.drop('Time_difference_seconds',axis=1)

    # 날짜 및 시간 열 설정
    synthetic_subset['Customer_registration_datetime'] = pd.to_datetime(
        synthetic_subset['Customer_registration_datetime'], errors = 'coerce'
    )
    
    # 조건에 따라 값을 조정
    synthetic_subset['Customer_registration_datetime'] = synthetic_subset['Customer_registration_datetime'].apply(
    lambda x: (min_date + pd.Timedelta(days=np.random.randint(1, 360))) if x < min_date else
                (regi_date - pd.Timedelta(days=np.random.randint(1, 360))) if x > regi_date else
                x
    )
    synthetic_subset['Account_creation_datetime'] = pd.to_datetime(
    synthetic_subset['Account_creation_datetime'], errors='coerce'
    )

    synthetic_subset['Account_creation_datetime'] = pd.to_datetime(
        synthetic_subset['Account_creation_datetime'], errors='coerce'
    ).where(
        synthetic_subset['Account_creation_datetime'] >= synthetic_subset['Customer_registration_datetime'],
        synthetic_subset['Customer_registration_datetime'] + pd.Timedelta(days=np.random.randint(1, 90))
    ).where(
        synthetic_subset['Account_creation_datetime'] < reference_date,
        reference_date - pd.Timedelta(days=np.random.randint(1, 30))
    )
    # 나머지 날짜 열 설정 (순서 상관 없음)
    date_columns = [
    'Last_atm_transaction_datetime',
    'Last_bank_branch_transaction_datetime',
    'Transaction_resumed_date',
    ]

    for col in date_columns:
        synthetic_subset[col] = pd.to_datetime(synthetic_subset[col], errors='coerce')
        synthetic_subset[col] = pd.to_datetime(synthetic_subset[col], errors='coerce').where(
            synthetic_subset[col] >= synthetic_subset['Account_creation_datetime'],
            synthetic_subset['Account_creation_datetime'] + pd.Timedelta(days=np.random.randint(1, 30))
        ).where(
            synthetic_subset[col] < reference_date,
            reference_date - pd.Timedelta(days=np.random.randint(1, 30)
        )
        )
    # 마지막으로 Transaction_Datetime 처리
    synthetic_subset['Transaction_Datetime'] = pd.to_datetime(synthetic_subset['Transaction_Datetime'], errors='coerce')
    synthetic_subset['Transaction_Datetime'] = pd.to_datetime(synthetic_subset['Transaction_Datetime'], errors='coerce').where(
        synthetic_subset['Transaction_Datetime'] >= synthetic_subset['Account_creation_datetime'],
        synthetic_subset['Account_creation_datetime'] + pd.Timedelta(days=np.random.randint(1, 30))
    ).where(
        synthetic_subset['Transaction_Datetime'] < reference_date,
        reference_date - pd.Timedelta(days=np.random.randint(1, 5))
    )


        # 생성된 데이터를 all_synthetic_data에 추가
    all_synthetic_data = pd.concat([all_synthetic_data, synthetic_subset], ignore_index=True)


# 최종 결과 확인
print("\nFinal All Synthetic Data Shape:", all_synthetic_data.shape)
all_synthetic_data.head()

100%|██████████| 13/13 [12:16<00:00, 56.65s/it]


Final All Synthetic Data Shape: (13000, 63)


,Customer_Birthyear,Customer_Gender,Customer_personal_identifier,Customer_identification_number,Customer_registration_datetime,Customer_credit_rating,Customer_flag_change_of_authentication_1,Customer_flag_change_of_authentication_2,Customer_flag_change_of_authentication_3,Customer_flag_change_of_authentication_4,Customer_rooting_jailbreak_indicator,Customer_mobile_roaming_indicator,Customer_VPN_Indicator,Customer_loan_type,Customer_flag_terminal_malicious_behavior_1,Customer_flag_terminal_malicious_behavior_2,Customer_flag_terminal_malicious_behavior_3,Customer_flag_terminal_malicious_behavior_4,Customer_flag_terminal_malicious_behavior_5,Customer_flag_terminal_malicious_behavior_6,Customer_inquery_atm_limit,Customer_increase_atm_limit,Account_account_number,Account_account_type,Account_creation_datetime,Account_initial_balance,Account_balance,Account_indicator_release_limit_excess,Account_amount_daily_limit,Account_indicator_Openbanking,Account_remaining_amount_daily_limit_exceeded,Account_release_suspention,Account_one_month_max_amount,Account_one_month_std_dev,Account_dawn_one_month_max_amount,Account_dawn_one_month_std_dev,Transaction_Datetime,Transaction_Amount,Channel,Operating_System,Error_Code,Transaction_Failure_Status,Type_General_Automatic,IP_Address,MAC_Address,Access_Medium,Location,Recipient_Account_Number,Transaction_num_connection_failure,Another_Person_Account,Distance,Unused_terminal_status,Last_atm_transaction_datetime,Last_bank_branch_transaction_datetime,Flag_deposit_more_than_tenMillion,Unused_account_status,Recipient_account_suspend_status,Number_of_transaction_with_the_account,Transaction_history_with_the_account,First_time_iOS_by_vulnerable_user,Fraud_Type,Transaction_resumed_date,Time_difference
0,1950,male,강지우,VxbxkP-MIxGYBA,2008-10-28 19:12:26,C,0,0,0,1,0,0,0,b,0,0,1,0,1,0,1,1,SlqMXMToTB,d,2008-12-01 19:12:26,-5650528,34810277,1,1000000,1,13617159,1,41174063,5422566,6278582,0,2019-08-13 16:35:13,62988260,Others,Windows,a,0,general,123.119.143.95,82:a7:32:c6:40:99,a,경상북도 영주시 이산면 36.809067 128.709962,CTJdVNiUfi,0,1,170.217057,1,2008-12-16 19:12:26,2008-12-02 19:12:26,0,1,1,0,1,0,m,2030-12-13 19:41:36,1970-01-01 00:01:38
1,1962,female,김영환,oYOHiy-JMIOltt,2003-03-12 21:28:24,C,1,1,0,1,0,0,0,c,0,0,0,0,0,0,0,1,xnzHCIusOV,b,2008-11-10 00:38:16,-5650528,17244970,1,2000000,1,7773336,1,171825295,0,35584400,0,2028-09-30 06:59:37,12506445,ATM,Windows,a,0,general,117.88.205.134,1e:cb:ed:ec:5e:b1,a,전라남도 보성군 웅치면 34.701456 127.002965,rgEYkTIUfY,0,1,115.733417,1,2019-12-17 01:33:29,2008-11-11 00:38:16,0,1,0,1,0,0,m,2035-04-18 04:40:45,1970-01-01 00:01:38
2,1974,male,박성호,YuMAdG-cURFSEF,2003-11-30 06:25:49,B,0,0,1,1,0,0,0,c,0,1,0,0,0,0,1,1,IzfnUGclik,b,2008-08-04 08:47:09,-3904076,26596841,0,50000000,1,9195036,0,7634673,35556090,3551763,11285430,2030-11-04 04:36:00,-5356198,Others,Windows,a,0,general,55.10.230.232,58:74:f3:e4:89:b1,a,대구광역시 동구 지묘동 35.944149 128.639497,HtjyZLNcfM,0,1,219.541605,1,2008-08-19 08:47:09,2008-08-05 08:47:09,1,0,1,0,0,0,m,2039-02-06 11:20:56,1970-01-01 00:01:38
3,1992,male,서지은,RaCeMo-htYjkES,2003-03-12 21:28:24,B,1,0,1,0,0,0,0,b,0,0,0,0,0,0,1,0,TTDYVBtULp,a,2012-12-21 03:05:31,-5650528,3771566,0,50000000,1,10831956,1,26627067,0,7459338,0,2039-02-06 12:31:54,-11003917,internet,Linux,a,0,automatic,68.205.87.169,6a:bb:3a:1e:5a:a4,a,경상북도 고령군 운수면 35.754624 128.295765,QLUSJSLbtL,0,1,408.643654,1,2015-04-05 14:26:35,2016-04-29 17:45:54,0,1,0,0,0,0,m,2018-04-04 06:53:00,1970-01-01 00:01:38
4,1986,female,허주원,BvmhII-bZkEbSm,2005-06-28 20:22:25,B,1,1,1,1,0,0,0,c,0,0,0,0,0,0,0,1,TqWBlueBVJ,a,2010-10-23 08:10:14,-749916,26102082,0,1000000,1,2461059,1,23416890,0,6572619,0,2027-07-18 12:05:00,4577406,mobile,Windows,a,0,automatic,144.43.137.21,e2:1d:3e:be:5d:89,a,충청남도 서천군 서천읍 36.107386 126.702284,GKKdmLIANt,0,1,270.341435,1,2012-10-25 13:29:46,2013-08-01 22:22:42,0,1,1,1,3,0,m,2039-02-06 11:20:56,1970-01-01 00:01:38
